# Actividad Grupal: Resolución de un Problema mediante Búsqueda Heurística A*

**Materia:** Toma de Decisiones  
**Actividad:** Grupal — Búsqueda Heurística

---

## Descripción del Problema

La empresa Amazon desea utilizar un robot para ordenar el inventario de su almacén. El almacén se representa como una cuadrícula **4×4**. El robot debe mover tres inventarios (M1, M2, M3) desde sus posiciones iniciales hasta las posiciones objetivo.

### Estado Inicial

```
     0    1    2    3
0  [ M1] [ # ] [   ] [ M3]
1  [   ] [ # ] [   ] [   ]
2  [ M2] [   ] [ R ] [   ]
3  [   ] [   ] [   ] [   ]
```

- **R** (Robot): posición inicial `[2,2]`
- **#** (Paredes): `[0,1]` y `[1,1]`
- **M1**: `[0,0]`
- **M2**: `[2,0]`
- **M3**: `[0,3]`

### Estado Objetivo

```
     0    1    2    3
0  [   ] [ # ] [   ] [   ]
1  [   ] [ # ] [   ] [   ]
2  [   ] [   ] [   ] [   ]
3  [   ] [ M3] [ M2] [ M1]
```

- **M1** → `[3,3]`
- **M2** → `[3,2]`
- **M3** → `[3,1]`

## 1. Importación de Librerías

Se utilizan únicamente librerías estándar de Python: `heapq` para la cola de prioridad del algoritmo A*, y `copy` para realizar copias profundas del estado.

In [ ]:
import heapq
import copy

## 2. Representación del Estado

El estado del problema se representa con:
- La posición del robot `(fila, columna)`.
- Las posiciones de cada inventario: M1, M2 y M3.
- Si el robot está cargando algún inventario (o `None` si no carga nada).

Las **paredes** (`#`) son fijas y se definen como una constante global. El coste real de cada acción es **1**.

In [ ]:
# Dimensiones de la cuadrícula
ROWS, COLS = 4, 4

# Posiciones fijas de las paredes
WALLS = {(0, 1), (1, 1)}

# Estado objetivo de los inventarios
GOAL_POSITIONS = {
    'M1': (3, 3),
    'M2': (3, 2),
    'M3': (3, 1)
}


class State:
    """
    Representa un estado en el espacio de búsqueda.

    Atributos:
        robot      : tupla (fila, columna) con la posición del robot.
        inventories: diccionario {'M1': (r,c), 'M2': (r,c), 'M3': (r,c)}
        carrying   : nombre del inventario que carga el robot ('M1'/'M2'/'M3') o None.
    """

    def __init__(self, robot, inventories, carrying=None):
        self.robot = robot
        self.inventories = inventories  # dict
        self.carrying = carrying

    def to_tuple(self):
        """Convierte el estado a una tupla hashable para usar como clave."""
        inv = tuple(sorted(self.inventories.items()))
        return (self.robot, inv, self.carrying)

    def is_goal(self):
        """Devuelve True si todos los inventarios están en sus posiciones objetivo."""
        return self.inventories == GOAL_POSITIONS and self.carrying is None

    def __repr__(self):
        return (f"Robot={self.robot}, Inventarios={self.inventories}, "
                f"Cargando={self.carrying}")

## 3. Función Heurística — Distancia Manhattan

La heurística utilizada es la **Distancia de Manhattan**, que estima el coste mínimo restante para llevar cada inventario a su posición objetivo.

Para cada inventario que aún no está en su destino, se suma la distancia Manhattan desde su posición actual hasta la posición objetivo:

$$h(n) = \sum_{i \in \{M1, M2, M3\}} \left( |r_i - r_{goal_i}| + |c_i - c_{goal_i}| \right)$$

Esta heurística es **admisible** (nunca sobreestima el coste real) porque el robot necesita al menos tantos pasos como la distancia Manhattan para llevar cada inventario a su destino.

In [ ]:
def manhattan_distance(pos1, pos2):
    """Calcula la distancia Manhattan entre dos posiciones (fila, columna)."""
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])


def heuristic(state):
    """
    Función heurística h(n): suma de las distancias Manhattan
    de cada inventario a su posición objetivo.
    """
    total = 0
    for item, goal_pos in GOAL_POSITIONS.items():
        current_pos = state.inventories[item]
        total += manhattan_distance(current_pos, goal_pos)
    return total

## 4. Generación de Acciones (Sucesores)

Desde cada estado, el robot puede realizar las siguientes acciones:

1. **Moverse** en 4 direcciones (arriba, abajo, izquierda, derecha), siempre que la celda destino esté dentro de la cuadrícula, no sea una pared, y no esté ocupada por un inventario que no está siendo cargado.
2. **Cargar** un inventario: si el robot está en la misma celda que un inventario y no está cargando nada.
3. **Descargar** un inventario: si el robot está cargando uno y la celda actual está libre (no hay otro inventario, no es pared).

Cada acción tiene coste **g = 1**.

In [ ]:
MOVES = {
    'arriba':    (-1,  0),
    'abajo':     ( 1,  0),
    'izquierda': ( 0, -1),
    'derecha':   ( 0,  1)
}


def get_successors(state):
    """
    Genera todos los estados sucesores posibles a partir del estado dado.

    Retorna una lista de tuplas (nuevo_estado, descripcion_accion).
    """
    successors = []
    r, c = state.robot

    # Posiciones ocupadas por inventarios que NO está cargando el robot
    blocked_by_inventory = {
        pos for item, pos in state.inventories.items()
        if item != state.carrying
    }

    # --- Acción: MOVER el robot ---
    for direction, (dr, dc) in MOVES.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS:
            if (nr, nc) not in WALLS and (nr, nc) not in blocked_by_inventory:
                new_inventories = dict(state.inventories)
                # Si lleva un inventario, lo mueve con él
                if state.carrying:
                    new_inventories[state.carrying] = (nr, nc)
                new_state = State((nr, nc), new_inventories, state.carrying)
                action = f"mover R fila{nr} columna{nc}"
                successors.append((new_state, action))

    # --- Acción: CARGAR inventario ---
    if state.carrying is None:
        for item, pos in state.inventories.items():
            if pos == state.robot:
                new_state = State(state.robot, dict(state.inventories), item)
                action = f"cargar R {item} fila{r} columna{c}"
                successors.append((new_state, action))

    # --- Acción: DESCARGAR inventario ---
    if state.carrying is not None:
        # Se puede descargar en la posición actual
        # La posición debe quedar libre (no hay otro inventario ya ahí)
        other_positions = {
            pos for item, pos in state.inventories.items()
            if item != state.carrying
        }
        if state.robot not in other_positions and state.robot not in WALLS:
            new_inventories = dict(state.inventories)
            new_inventories[state.carrying] = state.robot
            new_state = State(state.robot, new_inventories, None)
            action = (f"descargar R {state.carrying} "
                      f"fila{r} columna{c}")
            successors.append((new_state, action))

    return successors

## 5. Algoritmo A*

El algoritmo A* explora el espacio de estados usando una **lista abierta** (cola de prioridad por f(n) = g(n) + h(n)) y una **lista cerrada** (estados ya explorados).

En cada iteración:
1. Se extrae el nodo con menor `f(n)` de la lista abierta.
2. Si es el estado objetivo, se reconstruye el plan y se termina.
3. Se expande el nodo: se generan los sucesores y se añaden a la lista abierta si no han sido visitados con menor coste.

Se registra el contenido de la lista abierta y cerrada en cada iteración para análisis posterior.

In [ ]:
def astar(initial_state):
    """
    Ejecuta el algoritmo A* desde el estado inicial.

    Retorna:
        plan       : lista de strings con las acciones del plan solución.
        iterations : lista de dicts con el estado de listas abierta/cerrada por iteración.
    """
    # Nodo: (f, g, contador_desempate, estado, camino)
    h0 = heuristic(initial_state)
    counter = 0
    open_list = [(h0, 0, counter, initial_state, [])]
    heapq.heapify(open_list)

    # Diccionario de mejor coste g conocido para cada estado
    best_g = {initial_state.to_tuple(): 0}

    # Lista cerrada: estados ya expandidos
    closed_set = set()

    iterations = []  # Registro de iteraciones para análisis

    while open_list:
        f, g, _, current_state, path = heapq.heappop(open_list)

        state_key = current_state.to_tuple()

        if state_key in closed_set:
            continue
        closed_set.add(state_key)

        # Registrar iteración
        iterations.append({
            'f': f, 'g': g, 'h': f - g,
            'estado': repr(current_state),
            'cerrada_size': len(closed_set),
            'abierta_size': len(open_list)
        })

        # ¿Hemos llegado al objetivo?
        if current_state.is_goal():
            return path, iterations

        # Expandir sucesores
        for next_state, action in get_successors(current_state):
            next_key = next_state.to_tuple()
            if next_key in closed_set:
                continue
            new_g = g + 1  # Coste real de cada acción = 1
            if new_g < best_g.get(next_key, float('inf')):
                best_g[next_key] = new_g
                h = heuristic(next_state)
                f_new = new_g + h
                counter += 1
                heapq.heappush(
                    open_list,
                    (f_new, new_g, counter, next_state, path + [action])
                )

    return None, iterations  # No se encontró solución

## 6. Ejecución del Algoritmo

Se define el estado inicial con las posiciones descritas en el enunciado y se ejecuta A*. Al finalizar, se imprime:
- El **plan de acciones** en formato legible.
- Un **resumen estadístico** del proceso de búsqueda.

In [ ]:
# Definición del estado inicial
initial_state = State(
    robot=(2, 2),
    inventories={
        'M1': (0, 0),
        'M2': (2, 0),
        'M3': (0, 3)
    },
    carrying=None
)

print("Estado inicial:")
print(f"  {initial_state}")
print(f"\nEstado objetivo:")
for item, pos in GOAL_POSITIONS.items():
    print(f"  {item} → {pos}")

print("\n" + "="*60)
print("Ejecutando algoritmo A*...")
print("="*60)

plan, iterations = astar(initial_state)

if plan is not None:
    print(f"\n✓ Solución encontrada en {len(plan)} acciones.")
    print(f"  Nodos explorados (lista cerrada): {iterations[-1]['cerrada_size']}")
    print(f"  Iteraciones totales: {len(iterations)}")
else:
    print("\n✗ No se encontró solución.")

## 7. Plan de Acción — Secuencia de Pasos

A continuación se muestra la **secuencia completa de acciones** que el robot debe realizar para mover los tres inventarios desde el estado inicial al estado objetivo.

In [ ]:
if plan:
    print("\nPlan de acción (secuencia de pasos):")
    print("-" * 45)
    for i, action in enumerate(plan, 1):
        print(f"  Paso {i:3d}: {action}")
    print("-" * 45)
    print(f"  Total de pasos: {len(plan)}")

## 8. Análisis de las Primeras Iteraciones (Lista Abierta y Cerrada)

En esta sección se muestra el contenido resumido de la **lista cerrada** (nodos explorados) y la **lista abierta** (nodos pendientes) para las primeras iteraciones del algoritmo, con los valores `g(n)`, `h(n)` y `f(n)` de cada nodo expandido.

In [ ]:
print("Primeras 20 iteraciones del algoritmo A*:")
print(f"{'Iter':>5} {'g(n)':>6} {'h(n)':>6} {'f(n)':>6} {'|Cerrada|':>10} {'|Abierta|':>10}")
print("-" * 50)
for i, it in enumerate(iterations[:20], 1):
    print(f"{i:>5} {it['g']:>6} {it['h']:>6} {it['f']:>6} "
          f"{it['cerrada_size']:>10} {it['abierta_size']:>10}")

## 9. Visualización del Estado Inicial y Objetivo

Se imprime la cuadrícula 4×4 del estado inicial y del estado objetivo para facilitar la comprensión visual del problema.

In [ ]:
def print_grid(robot, inventories, carrying=None, title="Estado"):
    """Imprime la cuadrícula 4x4 del estado dado."""
    grid = [['   ' for _ in range(COLS)] for _ in range(ROWS)]

    for (r, c) in WALLS:
        grid[r][c] = ' # '
    for item, (r, c) in inventories.items():
        grid[r][c] = f' {item}'
    r, c = robot
    if carrying:
        grid[r][c] = f'R({carrying[1]})'
    else:
        grid[r][c] = ' R '

    print(f"\n{title}")
    print("     " + "  ".join(str(j) for j in range(COLS)))
    for i, row in enumerate(grid):
        print(f"  {i}  " + " ".join(cell.ljust(3) for cell in row))


print_grid(
    robot=(2, 2),
    inventories={'M1': (0, 0), 'M2': (2, 0), 'M3': (0, 3)},
    title="Estado Inicial"
)

print_grid(
    robot=(2, 2),  # posición final del robot no es obligatoria
    inventories=GOAL_POSITIONS,
    title="Estado Objetivo"
)

## 10. Conclusiones

- El algoritmo **A*** encuentra la solución **óptima** gracias al uso de la heurística de distancia Manhattan, que es admisible y consistente en este dominio.
- El **coste real** de cada acción (mover, cargar, descargar) es 1, lo que asegura que `g(n)` representa exactamente el número de pasos realizados.
- Las **paredes** actúan como obstáculos permanentes en la cuadrícula, obligando al robot a buscar rutas alternativas.
- La representación del estado incluye las posiciones del robot, de los inventarios y si el robot carga algún objeto, lo que permite capturar completamente la dinámica del problema.

---

## Referencias Bibliográficas

- Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4ª ed.). Pearson.
- Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A formal basis for the heuristic determination of minimum cost paths. *IEEE Transactions on Systems Science and Cybernetics*, 4(2), 100–107. https://doi.org/10.1109/TSSC.1968.300136
- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2009). *Introduction to Algorithms* (3ª ed.). MIT Press.